In [1]:
from pathlib import Path
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

CUTOFF = pd.Timestamp("2017-02-28")
WINDOW_START = CUTOFF - pd.Timedelta(days=89)

con = duckdb.connect()

labels_v2 = pd.read_csv(RAW_DIR / "train_v2.csv")
con.register("cohort_users", labels_v2[["msno"]])

print("Window start:", WINDOW_START.date())
print("Feature cutoff:", CUTOFF.date())
print("Cohort users:", len(labels_v2))

Window start: 2016-12-01
Feature cutoff: 2017-02-28
Cohort users: 970960


In [2]:
feature_sql = """
WITH parsed AS (
    SELECT
        t.msno,
        TRY_STRPTIME(
            t.transaction_date, '%Y%m%d'
        )::DATE AS transaction_day,
        TRY_CAST(t.payment_plan_days AS INTEGER) AS plan_days,
        TRY_CAST(t.is_auto_renew AS INTEGER) AS auto_renew,
        TRY_CAST(t.is_cancel AS INTEGER) AS cancel_flag
    FROM read_csv(?, all_varchar = true) AS t
    INNER JOIN cohort_users AS c
        ON t.msno = c.msno
),
windowed AS (
    SELECT *
    FROM parsed
    WHERE transaction_day BETWEEN CAST(? AS DATE)
                              AND CAST(? AS DATE)
)
SELECT
    msno,
    COUNT(*) AS transaction_rows_90d,
    COUNT(DISTINCT transaction_day) AS transaction_days_90d,
    COUNT(*) FILTER (
        WHERE cancel_flag = 1
    ) AS cancel_rows_90d,
    AVG(auto_renew) AS auto_renew_share_90d,
    AVG(plan_days) AS avg_plan_days_90d,
    DATE_DIFF(
        'day',
        MAX(transaction_day),
        CAST(? AS DATE)
    ) AS days_since_last_transaction_90d
FROM windowed
GROUP BY msno
"""

transaction_features = con.execute(
    feature_sql,
    [
        str(RAW_DIR / "transactions.csv"),
        WINDOW_START.date(),
        CUTOFF.date(),
        CUTOFF.date(),
    ],
).df()

print("Feature table shape:", transaction_features.shape)
transaction_features.drop(columns="msno").head()

Feature table shape: (943179, 7)


,transaction_rows_90d,transaction_days_90d,cancel_rows_90d,auto_renew_share_90d,avg_plan_days_90d,days_since_last_transaction_90d
0,4,3,1,1.0,30.0,14
1,3,3,0,1.0,30.0,24
2,4,3,1,1.0,30.0,21
3,3,3,0,1.0,30.0,17
4,3,3,0,1.0,30.0,2


In [3]:
assert transaction_features["msno"].is_unique

assert transaction_features["transaction_rows_90d"].ge(1).all()

assert (
    transaction_features["transaction_days_90d"]
    <= transaction_features["transaction_rows_90d"]
).all()

assert transaction_features["auto_renew_share_90d"].between(0, 1).all()

assert transaction_features[
    "days_since_last_transaction_90d"
].between(0, 89).all()

transaction_features.drop(columns="msno").describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_rows_90d,943179.0,2.792312,0.628168,1.0,3.0,3.0,3.0,17.0
transaction_days_90d,943179.0,2.768505,0.593188,1.0,3.0,3.0,3.0,11.0
cancel_rows_90d,943179.0,0.043050,0.208622,0.0,0.0,0.0,0.0,6.0
auto_renew_share_90d,943179.0,0.900842,0.295379,0.0,1.0,1.0,1.0,1.0
avg_plan_days_90d,943179.0,30.204682,4.045013,1.0,30.0,30.0,30.0,410.0
days_since_last_transaction_90d,943179.0,13.920791,10.811113,0.0,5.0,13.0,22.0,89.0


In [4]:
feature_table_v2 = labels_v2.merge(
    transaction_features,
    on="msno",
    how="left",
    validate="one_to_one",
    indicator="transaction_join",
)

feature_table_v2["has_transaction_90d"] = (
    feature_table_v2["transaction_join"] == "both"
).astype("int8")

feature_table_v2 = feature_table_v2.drop(
    columns="transaction_join"
)

count_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
]

feature_table_v2[count_columns] = (
    feature_table_v2[count_columns]
    .fillna(0)
    .astype("int64")
)

assert len(feature_table_v2) == len(labels_v2)
assert feature_table_v2["msno"].is_unique

assert (
    feature_table_v2["cancel_rows_90d"]
    <= feature_table_v2["transaction_rows_90d"]
).all()

print("Feature table shape:", feature_table_v2.shape)
print(
    "Users without transactions in the window:",
    (feature_table_v2["has_transaction_90d"] == 0).sum(),
)

Feature table shape: (970960, 9)
Users without transactions in the window: 27781


In [5]:
transaction_coverage_summary = (
    feature_table_v2
    .groupby("has_transaction_90d")
    .agg(
        users=("msno", "size"),
        churned_users=("is_churn", "sum"),
        churn_rate=("is_churn", "mean"),
    )
    .reset_index()
)

transaction_coverage_summary["user_share"] = (
    transaction_coverage_summary["users"]
    / len(feature_table_v2)
)

transaction_coverage_summary

,has_transaction_90d,users,churned_users,churn_rate,user_share
0,0,27781,23272,0.837695,0.028612
1,1,943179,64058,0.067917,0.971388


In [6]:
import json

processed_dir = PROJECT_ROOT / "data" / "processed"
report_dir = PROJECT_ROOT / "reports"
sql_dir = PROJECT_ROOT / "sql"

for directory in [processed_dir, report_dir, sql_dir]:
    directory.mkdir(parents=True, exist_ok=True)

feature_path = (
    processed_dir
    / f"candidate_transaction_features_v2_cutoff_{CUTOFF:%Y%m%d}.parquet"
)

con.from_df(feature_table_v2).write_parquet(str(feature_path))

(sql_dir / "transaction_features_90d.sql").write_text(
    feature_sql.strip() + "\n",
    encoding="utf-8",
)

transaction_coverage_summary.to_csv(
    report_dir / "transaction_feature_coverage.csv",
    index=False,
)

transaction_features.drop(columns="msno").describe().T.to_csv(
    report_dir / "transaction_feature_statistics.csv",
    index_label="feature",
)

feature_config = {
    "status": "provisional",
    "transaction_source": "transactions.csv",
    "cohort_source": "train_v2.csv",
    "window_start": str(WINDOW_START.date()),
    "cutoff": str(CUTOFF.date()),
    "window_days": 90,
    "label_period_verified": False,
    "no_transaction_policy": {
        "counts": "zero",
        "averages_and_recency": "missing",
    },
}

(report_dir / "transaction_feature_config.json").write_text(
    json.dumps(feature_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Candidate features, SQL, configuration, and reports saved.")

Candidate features, SQL, configuration, and reports saved.
